In [33]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans


df = pd.read_csv("/workspaces/k-means-repo/data/raw/Mobile Reviews Sentiment null.csv")

print("shape \n", df.shape)

print("---------------------------------")

print("first 5 rows \n", df.head())

print("---------------------------------")


print("info \n", df.info())

print("---------------------------------")

print("Null count \n", df.isnull().sum())

print("---------------------------------")

print("Duplicates count \n", df.duplicated().sum())


shape 
 (50000, 22)
---------------------------------
first 5 rows 
    review_id      customer_name  age     brand          model  price_usd  \
0          1      Aryan Maharaj   45    Realme  Realme 12 Pro     337.31   
1          2  Davi Miguel Sousa   18    Realme  Realme 12 Pro     307.78   
2          3        Pahal Balay   27    Google        Pixel 6     864.53   
3          4       David Guzman   19    Xiaomi  Redmi Note 13     660.94   
4          5          Yago Leão   38  Motorola        Edge 50     792.13   

  price_local currency  exchange_rate_to_usd  rating  ...    language  \
0   ₹27996.73      INR                 83.00     2.0  ...       Hindi   
1   R$1754.35      BRL                  5.70     4.0  ...  Portuguese   
2   ₹71755.99      INR                 83.00     4.0  ...       Hindi   
3  د.إ2425.65      AED                  3.67     3.0  ...     English   
4   R$4515.14      BRL                  5.70     3.0  ...  Portuguese   

  review_date verified_purchase bat

In [34]:
import re

# drop review id and customer name 
df = df.drop(columns = ['review_id', 'customer_name'])


# fill price_usd nulls with mean per brand
df['price_usd'] = df.groupby('brand')['price_usd'].transform(lambda x: x.fillna(x.mean()))

# fill rating nulls with mean per brand
df['rating'] = df.groupby('model')['rating'].transform(lambda x: x.fillna(x.mean()))

# fill source nulls with mode
df['source'] = df['source'].fillna(df['source'].mode()[0])

# fill sentiment by average of all ratings column

rating_cols = ['battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating', 'helpful_votes']
df['avg_rating'] = df[rating_cols].mean(axis=1)

def assign_sentiment(row):
    if pd.notnull(row['sentiment']): 
        return row['sentiment']
    if(row['avg_rating'] < 3) :
        return 'Negative'
    elif(row['avg_rating'] == 3) : 
        return 'Neutral'
    else:
        return 'Positive'

df['sentiment'] = df.apply(assign_sentiment, axis = 1)

# drop avg_rating
df = df.drop(columns = 'avg_rating')

def split_price(val):
    if pd.isnull(val):
        return None, None
    val = str(val).strip()
    match = re.match(r'^([^\d]+)([\d,\.]+)$', val)
    if match:
        return match.group(1).strip(), float(match.group(2).replace(',', ''))
    return None, None

df[['currency_symbol', 'price_local_value']] = df['price_local'].apply(lambda x: pd.Series(split_price(x)))

print("first 5 rows of price \n", df[['price_local', 'currency_symbol', 'price_local_value']].head())

# Check UAE rows specifically
print("Check UAE rows specifically \n", df[df['currency'] == 'AED'][['price_local', 'currency_symbol', 'price_local_value']].head())

df['price_local_value'] = df['price_usd'] * df['exchange_rate_to_usd']

# drop currency, currency_symbol, exchange_rate_to_usd, price_local 
df = df.drop(columns = ['currency', 'currency_symbol', 'exchange_rate_to_usd', 'price_local'])

print("shape \n", df.shape)

print("---------------------------------")

print("first 5 rows \n", df.head())

print("---------------------------------")


print("info \n", df.info())

print("---------------------------------")

print("Null count \n", df.isnull().sum())

print("---------------------------------")

print("Duplicates count \n", df.duplicated().sum())





first 5 rows of price 
   price_local currency_symbol  price_local_value
0   ₹27996.73               ₹           27996.73
1   R$1754.35              R$            1754.35
2   ₹71755.99               ₹           71755.99
3  د.إ2425.65             د.إ            2425.65
4   R$4515.14              R$            4515.14
Check UAE rows specifically 
    price_local currency_symbol  price_local_value
3   د.إ2425.65             د.إ            2425.65
6   د.إ1541.88             د.إ            1541.88
27   د.إ867.51             د.إ             867.51
58   د.إ775.21             د.إ             775.21
72  د.إ4469.95             د.إ            4469.95
shape 
 (50000, 18)
---------------------------------
first 5 rows 
    age     brand          model  price_usd  rating sentiment country  \
0   45    Realme  Realme 12 Pro     337.31     2.0  Negative   India   
1   18    Realme  Realme 12 Pro     307.78     4.0  Positive  Brazil   
2   27    Google        Pixel 6     864.53     4.0  Positive   Indi

In [35]:
# convert data types

df['rating'] = df['rating'].astype(int)

df["verified_purchase"] = df["verified_purchase"].astype(int)


print("info \n", df.info())

print("---------------------------------")

print("first 5 rows \n", df.head())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  50000 non-null  int64  
 1   brand                50000 non-null  str    
 2   model                50000 non-null  str    
 3   price_usd            50000 non-null  float64
 4   rating               50000 non-null  int64  
 5   sentiment            50000 non-null  str    
 6   country              50000 non-null  str    
 7   language             50000 non-null  str    
 8   review_date          50000 non-null  str    
 9   verified_purchase    50000 non-null  int64  
 10  battery_life_rating  50000 non-null  int64  
 11  camera_rating        50000 non-null  int64  
 12  performance_rating   50000 non-null  int64  
 13  design_rating        50000 non-null  int64  
 14  display_rating       50000 non-null  int64  
 15  helpful_votes        50000 non-null  int64  
 1

In [36]:
# convert review date into three columns -> month, day and year

df['review_date'] = pd.to_datetime(df['review_date'])

df['review_month'] = df['review_date'].dt.month
df['review_day'] = df['review_date'].dt.day
df['review_year'] = df['review_date'].dt.year


df = df.drop(columns = 'review_date')

print("info \n", df.info())

print("---------------------------------")

print("first 5 rows \n", df.head())

print("data types \n", df.dtypes)

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  50000 non-null  int64  
 1   brand                50000 non-null  str    
 2   model                50000 non-null  str    
 3   price_usd            50000 non-null  float64
 4   rating               50000 non-null  int64  
 5   sentiment            50000 non-null  str    
 6   country              50000 non-null  str    
 7   language             50000 non-null  str    
 8   verified_purchase    50000 non-null  int64  
 9   battery_life_rating  50000 non-null  int64  
 10  camera_rating        50000 non-null  int64  
 11  performance_rating   50000 non-null  int64  
 12  design_rating        50000 non-null  int64  
 13  display_rating       50000 non-null  int64  
 14  helpful_votes        50000 non-null  int64  
 15  source               50000 non-null  str    
 1

In [37]:
# save cleaned data

df.to_csv('/workspaces/k-means-repo/data/processed/cleaned_data_before_encoding.csv', index=False)

In [38]:
# Encode text columns

df = pd.get_dummies(df, columns=['brand', 'model', 'country', 'language', 'source'])

sentiment_map = {
    'Negative': 0,
    'Neutral': 1,
    'Positive': 2
}
df['sentiment'] = df['sentiment'].map(sentiment_map)

print("shape \n", df.shape)

print("---------------------------------")

print("first 5 rows \n", df.head())

print("---------------------------------")


print("info \n", df.info())



shape 
 (50000, 61)
---------------------------------
first 5 rows 
    age  price_usd  rating  sentiment  verified_purchase  battery_life_rating  \
0   45     337.31       2          0                  1                    1   
1   18     307.78       4          2                  1                    3   
2   27     864.53       4          2                  1                    3   
3   19     660.94       3          2                  0                    1   
4   38     792.13       3          1                  1                    3   

   camera_rating  performance_rating  design_rating  display_rating  ...  \
0              1                   3              2               1  ...   
1              2                   4              3               2  ...   
2              5                   3              2               4  ...   
3              3                   2              1               2  ...   
4              3                   2              2               1  .

In [39]:
# save cleaned data before scaling

df.to_csv('/workspaces/k-means-repo/data/processed/cleaned_data_before_scaling.csv', index=False)

In [40]:
# Scaling the data

# If max is very far from 75% → outliers exist → StandardScaler
# If max is close to 75% → no big outliers → MinMaxScaler is fin
# choose MinMaxScaler if No significant outliers, want 0 to 1 range
# choose StandardScaler if Data has outliers, want balanced distances 

print(df['price_usd'].describe())

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df)

count    50000.000000
mean       689.798406
std        307.332102
min        180.020000
25%        449.753290
50%        640.805000
75%        900.504534
max       1499.890000
Name: price_usd, dtype: float64


In [41]:
df_scaled_df = pd.DataFrame(df_scaled, columns=df.columns)
df_scaled_df.to_csv('/workspaces/k-means-repo/data/processed/cleaned_data.csv', index=False)